In [1]:
import os
from pathlib import Path
import glob
import cv2
import json

In [2]:
BASE_DIR = Path("./MVTEC")
NEW_DIR = Path("./MVTEC_det")
NEW_DIR.mkdir(exist_ok=True)

class_names = ["bottle", "cable", "capsule", "carpet", "grid", "hazelnut", "leather", "metal_nut", "pill", "screw", "tile", "toothbrush", "transistor", "wood", "zipper"]
base_split = ["train", "test"]
base_cate = ["good", "defect"]

In [3]:
def normalize_bbox(bbox, width, height):
    """
    Normalize bounding box to the range [0, 100].
    """
    return [
        int((bbox[0] / width) * 100),  # x_min
        int((bbox[1] / height) * 100),  # y_min
        int((bbox[2] / width) * 100),  # x_max
        int((bbox[3] / height) * 100),  # y_max
    ]

In [ ]:
mvtech_ad_data_for_regression = []

for class_name in class_names:
    print(f"Processing {class_name}")
    class_dir = Path(BASE_DIR) / class_name
    new_class_dir = NEW_DIR / class_name
    new_class_dir.mkdir(exist_ok=True)
    
    for split in base_split:
        if split == "train":
            cmd = f"cp -r {class_dir / split}/good {new_class_dir}/"
            os.system(cmd)

            image_paths = glob.glob(f"{new_class_dir}/good/*.png")
            for image_path in image_paths:
                img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
                # bbox is the image size
                bbox = [0, 0, img.shape[1], img.shape[0]]
                mvtech_ad_data_for_regression.append({
                    "image_path": image_path,
                    "bbox": bbox,
                    "class": class_name,
                    "is_broken": False,
                    "height": img.shape[0],
                    "width": img.shape[1]
                })

        else:
            new_broken_dir = new_class_dir / "broken"
            new_broken_dir.mkdir(exist_ok=True)

            last_good_idx = int(sorted(glob.glob(f'{class_dir}/train/good/*.png'))[-1].split('/')[-1].split('.')[0])
            good_images = sorted(glob.glob(f"{class_dir / split}/good/*.png"))
            for i, good_image in enumerate(good_images):
                new_name = good_image.split("/")[:-1] + [f"{last_good_idx+i+1:03d}.png"]
                new_name.remove("test")
                new_name = "/".join(new_name)
                new_name = new_name.replace("MVTEC", "MVTEC_det")
                cmd = f"cp {good_image} {new_name}"
                os.system(cmd)

                img = cv2.imread(new_name, cv2.IMREAD_GRAYSCALE)
                bbox = [0, 0, img.shape[1], img.shape[0]]
                mvtech_ad_data_for_regression.append({
                    "image_path": new_name,
                    "bbox": bbox,
                    "class": class_name,
                    "is_broken": False,
                    "height": img.shape[0],
                    "width": img.shape[1]
                })

            broken_idx = 0
            broken_cates = [x.split("/")[-1] for x in glob.glob(f"{class_dir / split}/*")]
            broken_cates.remove("good")
            for broken_cate in broken_cates:
                image_paths = sorted(glob.glob(f"{class_dir / split}/{broken_cate}/*.png"))
                for image_path in image_paths:
                    new_name = image_path.split("/")[:-1] + [f"{broken_idx:03d}.png"]
                    new_name.remove("test")
                    new_name = "/".join(new_name)
                    new_name = new_name.replace(broken_cate, "broken")
                    new_name = new_name.replace("MVTEC", "MVTEC_det")
                    cmd = f"cp {image_path} {new_name}"
                    os.system(cmd)

                    mask_path = image_path.replace("test", "ground_truth").replace(".png", "_mask.png")
                    mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
                    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
                    if not contours:
                        raise ValueError("No contours found")
                    x, y, w, h = cv2.boundingRect(contours[0])
                    bbox = [x, y, x + w, y + h]
                    mvtech_ad_data_for_regression.append({
                        "image_path": new_name,
                        "bbox": bbox,
                        "class": class_name,
                        "is_broken": True,
                        "height": mask.shape[0],
                        "width": mask.shape[1]
                    })

                    broken_idx += 1
    

In [ ]:
len(mvtech_ad_data_for_regression)

In [6]:
# shuffle the data

import random
random.seed(0)

random.shuffle(mvtech_ad_data_for_regression)

In [7]:
for data in mvtech_ad_data_for_regression:
    data["image_path"] = data["image_path"].replace("MVTEC_det", ".")
    data["bbox"] = normalize_bbox(
        bbox=data["bbox"],
        width=data["width"],
        height=data["height"]
    )

In [8]:
with open("./MVTEC_det/mvtech_ad_data_for_regression.json", "w") as f:
    json.dump(mvtech_ad_data_for_regression, f, indent=4)

In [9]:
# Move all the images to the images folder

os.makedirs("./MVTEC_det/images", exist_ok=True)

for class_name in class_names:
    cmd = f"mv MVTEC_det/{class_name} MVTEC_det/images/"
    os.system(cmd)

In [10]:
# split data into train test val
train_ratio = 0.7
val_ratio = 0.1
test_ratio = 0.2

train_data = mvtech_ad_data_for_regression[:int(len(mvtech_ad_data_for_regression) * train_ratio)]
val_data = mvtech_ad_data_for_regression[int(len(mvtech_ad_data_for_regression) * train_ratio):int(len(mvtech_ad_data_for_regression) * (train_ratio + val_ratio))]
test_data = mvtech_ad_data_for_regression[int(len(mvtech_ad_data_for_regression) * (train_ratio + val_ratio)):]



with open("./MVTEC_det/train_data.json", "w") as f:
    json.dump(train_data, f, indent=4)

with open("./MVTEC_det/val_data.json", "w") as f:
    json.dump(val_data, f, indent=4)

with open("./MVTEC_det/test_data.json", "w") as f:
    json.dump(test_data, f, indent=4)

In [11]:
# Statistic the data balance of the train, val, test set between classes and between good and defect

train_class_count = {class_name: 0 for class_name in class_names}
train_good_defect_count = {"good": 0, "defect": 0}

for data in train_data:
    train_class_count[data["class"]] += 1
    if data["is_broken"]:
        train_good_defect_count["defect"] += 1
    else:
        train_good_defect_count["good"] += 1

val_class_count = {class_name: 0 for class_name in class_names}
val_good_defect_count = {"good": 0, "defect": 0}

for data in val_data:
    val_class_count[data["class"]] += 1
    if data["is_broken"]:
        val_good_defect_count["defect"] += 1
    else:
        val_good_defect_count["good"] += 1

test_class_count = {class_name: 0 for class_name in class_names}
test_good_defect_count = {"good": 0, "defect": 0}

for data in test_data:
    test_class_count[data["class"]] += 1
    if data["is_broken"]:
        test_good_defect_count["defect"] += 1
    else:
        test_good_defect_count["good"] += 1


In [ ]:
# plot the data balance

import matplotlib.pyplot as plt

fig, ax = plt.subplots(3, 2, figsize=(20, 20))

ax[0, 0].bar(train_class_count.keys(), train_class_count.values())
ax[0, 0].set_title("Train Class Count")
ax[0, 0].set_xlabel("Class")

ax[0, 1].bar(val_class_count.keys(), val_class_count.values())
ax[0, 1].set_title("Val Class Count")
ax[0, 1].set_xlabel("Class")

ax[1, 0].bar(test_class_count.keys(), test_class_count.values())
ax[1, 0].set_title("Test Class Count")
ax[1, 0].set_xlabel("Class")

ax[1, 1].bar(train_good_defect_count.keys(), train_good_defect_count.values())
ax[1, 1].set_title("Train Good Defect Count")
ax[1, 1].set_xlabel("Good/Defect")

ax[2, 0].bar(val_good_defect_count.keys(), val_good_defect_count.values())
ax[2, 0].set_title("Val Good Defect Count")
ax[2, 0].set_xlabel("Good/Defect")

ax[2, 1].bar(test_good_defect_count.keys(), test_good_defect_count.values())
ax[2, 1].set_title("Test Good Defect Count")
ax[2, 1].set_xlabel("Good/Defect")

